In [14]:
import pandas as pd
import numpy as np

In [15]:
df = pd.read_csv("../data/MIG_features.csv", parse_dates=["date"])
df = df.set_index("date")
df.head()


,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,expected_closing,...,roll_7,roll_30,roll_90,month,quarter,day_of_week,rain_lag_1,temp_lag_1,behavior_chaotic,behavior_conservative
date,,,,,,,,,,,,,,,,,,,,,
2022-03-31,SITE_001,CEM_III,46.10,19.44,0.00,19.44,0.00,4.55,20.20,0.00,...,23.180000,29.936667,31.357333,3,1,3,5.82,13.85,False,False
2022-04-01,SITE_001,CEM_II,31.10,31.10,0.00,38.53,7.43,7.13,28.90,7.43,...,25.462857,30.973333,31.319111,4,2,4,4.55,20.20,False,False
2022-04-02,SITE_001,CEM_II,0.00,0.00,7.43,44.69,52.12,11.28,21.26,52.12,...,20.220000,29.862000,30.816222,4,2,5,7.13,28.90,False,False
2022-04-03,SITE_001,CEM_III,0.00,0.00,52.12,22.88,75.00,2.19,15.64,75.00,...,16.820000,28.259000,30.386333,4,2,6,11.28,21.26,False,False
2022-04-04,SITE_001,CEM_II,46.06,46.06,75.00,23.72,52.66,1.19,18.86,52.66,...,19.890000,28.176667,30.529667,4,2,0,2.19,15.64,False,False


In [16]:
df = df.sort_values(["site_id", "date"])
df.head()


,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,expected_closing,...,roll_7,roll_30,roll_90,month,quarter,day_of_week,rain_lag_1,temp_lag_1,behavior_chaotic,behavior_conservative
date,,,,,,,,,,,,,,,,,,,,,
2022-03-31,SITE_001,CEM_III,46.10,19.44,0.00,19.44,0.00,4.55,20.20,0.00,...,23.180000,29.936667,31.357333,3,1,3,5.82,13.85,False,False
2022-04-01,SITE_001,CEM_II,31.10,31.10,0.00,38.53,7.43,7.13,28.90,7.43,...,25.462857,30.973333,31.319111,4,2,4,4.55,20.20,False,False
2022-04-02,SITE_001,CEM_II,0.00,0.00,7.43,44.69,52.12,11.28,21.26,52.12,...,20.220000,29.862000,30.816222,4,2,5,7.13,28.90,False,False
2022-04-03,SITE_001,CEM_III,0.00,0.00,52.12,22.88,75.00,2.19,15.64,75.00,...,16.820000,28.259000,30.386333,4,2,6,11.28,21.26,False,False
2022-04-04,SITE_001,CEM_II,46.06,46.06,75.00,23.72,52.66,1.19,18.86,52.66,...,19.890000,28.176667,30.529667,4,2,0,2.19,15.64,False,False


In [8]:
sites = df["site_id"].unique()
sites


array(['SITE_001', 'SITE_002', 'SITE_003', 'SITE_004', 'SITE_005',
       'SITE_006', 'SITE_007', 'SITE_008', 'SITE_009', 'SITE_010',
       'SITE_011', 'SITE_012', 'SITE_013', 'SITE_014', 'SITE_015',
       'SITE_016', 'SITE_017', 'SITE_018', 'SITE_019', 'SITE_020',
       'SITE_021', 'SITE_022', 'SITE_023', 'SITE_024', 'SITE_025',
       'SITE_026', 'SITE_027', 'SITE_028', 'SITE_029', 'SITE_030'],
      dtype=object)

In [9]:
feature_cols = [
    'lag_1','lag_7','lag_14','lag_30',
    'roll_7','roll_30','roll_90',
    'month','quarter','day_of_week',
    'rain_lag_1','temp_lag_1',
    'behavior_chaotic','behavior_conservative'
]

target_col = "consumed_tonnes"


In [10]:
# Choose a split date
split_date = "2023-01-01"

train = df[df.index < split_date]
test = df[df.index >= split_date]

train.shape, test.shape


((8280, 30), (21930, 30))

Create per site datasets.

In [11]:
site_datasets = {}

for site in sites:
    site_train = train[train["site_id"] == site]
    site_test = test[test["site_id"] == site]

    X_train = site_train[feature_cols]
    y_train = site_train[target_col]

    X_test = site_test[feature_cols]
    y_test = site_test[target_col]

    site_datasets[site] = {
        "X_train": X_train,
        "y_train": y_train,
        "X_test": X_test,
        "y_test": y_test
    }

site_datasets["SITE_001"]["X_train"].head()


,lag_1,lag_7,lag_14,lag_30,roll_7,roll_30,roll_90,month,quarter,day_of_week,rain_lag_1,temp_lag_1,behavior_chaotic,behavior_conservative
date,,,,,,,,,,,,,,
2022-03-31,13.38,53.87,20.99,41.12,23.180000,29.936667,31.357333,3,1,3,5.82,13.85,False,False
2022-04-01,19.44,15.12,36.86,0.00,25.462857,30.973333,31.319111,4,2,4,4.55,20.20,False,False
2022-04-02,31.10,36.70,42.66,33.34,20.220000,29.862000,30.816222,4,2,5,7.13,28.90,False,False
2022-04-03,0.00,23.80,0.00,48.09,16.820000,28.259000,30.386333,4,2,6,11.28,21.26,False,False
2022-04-04,0.00,24.57,38.81,48.53,19.890000,28.176667,30.529667,4,2,0,2.19,15.64,False,False


Optional: Scale features (for ML models)

In [12]:
from sklearn.preprocessing import StandardScaler

scalers = {}

for site in sites:
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(site_datasets[site]["X_train"])
    X_test_scaled = scaler.transform(site_datasets[site]["X_test"])

    scalers[site] = scaler

    site_datasets[site]["X_train_scaled"] = X_train_scaled
    site_datasets[site]["X_test_scaled"] = X_test_scaled


Save the per-site datasets. This lets you load the datasets directly in Notebook 05 without repeating all the prep.

In [13]:
import pickle

with open("../data/site_datasets.pkl", "wb") as f:
    pickle.dump(site_datasets, f)
